# 訂單效能分析 — Summary Dashboard

整合 Layer 1a/1b/2/3 的分析結果，提供 executive summary。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

sys_flags = pd.read_csv('../data/system_anomaly_flags.csv')
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')
df = df.merge(sys_flags, on='order_id')
df = df.merge(usr_flags[['order_id', 'is_user_anomaly']], on='order_id')

print(f"Total orders: {len(df)}")
print(f"System anomalies: {df['is_system_anomaly'].sum()}")
print(f"User anomalies: {df['is_user_anomaly'].sum()}")
print(f"Normal orders: {(~(df['is_system_anomaly'] | df['is_user_anomaly'])).sum()}")


Total orders: 30000
System anomalies: 1576
User anomalies: 1459
Normal orders: 27033


## Executive Summary

本分析對 30,000 筆半導體製造訂單進行四層效能剖析。

In [2]:
# Build summary metrics
normal = df[~(df['is_system_anomaly'] | df['is_user_anomaly'])].copy()
anomaly_total = df['is_system_anomaly'].sum() + df['is_user_anomaly'].sum() - (df['is_system_anomaly'] & df['is_user_anomaly']).sum()

# Phase durations for normal orders
normal['est_queue'] = normal['queue_duration_seconds']
normal['est_db'] = normal['db_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_device'] = normal['device_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_inner'] = normal['inner_processing_duration_avg_seconds'] * normal['file_count'] / PARALLELISM

phase_means = normal[['est_queue', 'est_db', 'est_device', 'est_inner']].mean()
total_phase = phase_means.sum()
phase_pct = (phase_means / total_phase * 100).round(1)

print("=== EXECUTIVE SUMMARY ===")
print(f"\n1. 資料概觀")
print(f"   - 總訂單數: {len(df):,}")
print(f"   - 異常訂單: {anomaly_total:,} ({100*anomaly_total/len(df):.1f}%)")
print(f"   - 正常訂單: {len(normal):,}")
print(f"   - 平均耗時 (正常): {normal['total_duration_seconds'].mean():.0f}s (median: {normal['total_duration_seconds'].median():.0f}s)")

print(f"\n2. 系統異常 (Layer 1a)")
print(f"   - {df['is_system_anomaly'].sum()} 筆系統異常 ({100*df['is_system_anomaly'].mean():.1f}%)")

print(f"\n3. User 行為異常 (Layer 1b)")
print(f"   - {df['is_user_anomaly'].sum()} 筆行為異常 ({100*df['is_user_anomaly'].mean():.1f}%)")

print(f"\n4. 正常訂單瓶頸 (Layer 2)")
print(f"   - Queue: {phase_pct['est_queue']:.1f}% (avg {phase_means['est_queue']:.0f}s)")
print(f"   - DB: {phase_pct['est_db']:.1f}% (avg {phase_means['est_db']:.0f}s)")
print(f"   - Device: {phase_pct['est_device']:.1f}% (avg {phase_means['est_device']:.0f}s)")
print(f"   - Inner: {phase_pct['est_inner']:.1f}% (avg {phase_means['est_inner']:.0f}s)")
print(f"   - 主要瓶頸: {phase_pct.idxmax().replace('est_', '').upper()}")


=== EXECUTIVE SUMMARY ===

1. 資料概觀
   - 總訂單數: 30,000
   - 異常訂單: 2,967 (9.9%)
   - 正常訂單: 27,033
   - 平均耗時 (正常): 608s (median: 246s)

2. 系統異常 (Layer 1a)
   - 1576 筆系統異常 (5.3%)

3. User 行為異常 (Layer 1b)
   - 1459 筆行為異常 (4.9%)

4. 正常訂單瓶頸 (Layer 2)
   - Queue: 2.3% (avg 14s)
   - DB: 26.0% (avg 162s)
   - Device: 52.1% (avg 324s)
   - Inner: 19.6% (avg 122s)
   - 主要瓶頸: DEVICE


In [3]:
# Dashboard composite chart
fig = plt.figure(figsize=(20, 16))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: Overall duration distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['total_duration_seconds'].clip(upper=5000), bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(x=df['total_duration_seconds'].median(), color='red', linestyle='--', label=f"Median: {df['total_duration_seconds'].median():.0f}s")
ax1.set_title('Overall Duration Distribution')
ax1.set_xlabel('Total Duration (seconds)')
ax1.legend(fontsize=8)

# Panel 2: Anomaly breakdown pie
ax2 = fig.add_subplot(gs[0, 1])
sys_only = (df['is_system_anomaly'] & ~df['is_user_anomaly']).sum()
usr_only = (~df['is_system_anomaly'] & df['is_user_anomaly']).sum()
both = (df['is_system_anomaly'] & df['is_user_anomaly']).sum()
normal_count = len(normal)
sizes = [normal_count, sys_only, usr_only, both]
labels_pie = [f'Normal\n{normal_count:,}', f'System\n{sys_only:,}', f'User\n{usr_only:,}', f'Both\n{both:,}']
colors_pie = ['#2ecc71', '#e74c3c', '#f39c12', '#8e44ad']
ax2.pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%', startangle=90)
ax2.set_title('Order Classification')

# Panel 3: Phase breakdown overall
ax3 = fig.add_subplot(gs[0, 2])
phase_labels = ['Queue', 'DB', 'Device', 'Inner']
phase_values = [phase_pct['est_queue'], phase_pct['est_db'], phase_pct['est_device'], phase_pct['est_inner']]
colors_phase = ['#f39c12', '#3498db', '#e74c3c', '#2ecc71']
bars = ax3.bar(phase_labels, phase_values, color=colors_phase)
ax3.set_title('Phase Proportion (Normal Orders)')
ax3.set_ylabel('Proportion (%)')
for bar, val in zip(bars, phase_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%', ha='center', fontsize=9)

# Panel 4: Duration by file_count group
ax4 = fig.add_subplot(gs[1, 0:2])
bins = [0, 50, 300, 1000, 2000, 5000]
fc_labels = ['<50', '50-300', '300-1000', '1000-2000', '2000+']
normal['fc_group'] = pd.cut(normal['file_count'], bins=bins, labels=fc_labels, right=True)
group_data = normal.groupby('fc_group', observed=True)['total_duration_seconds'].agg(['median', 'mean', 'count'])
x = range(len(group_data))
ax4.bar([i-0.15 for i in x], group_data['median'], width=0.3, label='Median', color='steelblue')
ax4.bar([i+0.15 for i in x], group_data['mean'], width=0.3, label='Mean', color='coral')
ax4.set_xticks(x)
ax4.set_xticklabels(group_data.index)
ax4.set_title('Duration by File Count Group (Normal Orders)')
ax4.set_xlabel('File Count Group')
ax4.set_ylabel('Duration (seconds)')
ax4.legend()
for i, cnt in enumerate(group_data['count']):
    ax4.text(i, group_data['mean'].iloc[i] + 20, f'n={cnt}', ha='center', fontsize=8)

# Panel 5: Top 10 slow devices
ax5 = fig.add_subplot(gs[1, 2])
device_perf = normal.groupby('device_id')['device_duration_avg_seconds'].median().nlargest(10)
ax5.barh(range(len(device_perf)), device_perf.values, color='#e74c3c')
ax5.set_yticks(range(len(device_perf)))
ax5.set_yticklabels(device_perf.index, fontsize=7)
ax5.set_title('Top 10 Slowest Devices')
ax5.set_xlabel('Median Device Duration (s)')
ax5.invert_yaxis()

# Panel 6: Duration timeline
ax6 = fig.add_subplot(gs[2, :])
ax6.scatter(df['order_created_at'], df['total_duration_seconds'], alpha=0.05, s=3, c='gray')
anomaly_mask = df['is_system_anomaly'] | df['is_user_anomaly']
ax6.scatter(df.loc[anomaly_mask, 'order_created_at'], df.loc[anomaly_mask, 'total_duration_seconds'],
            alpha=0.3, s=8, c='red', label='Anomaly')
ax6.set_title('Order Duration Timeline')
ax6.set_xlabel('Time')
ax6.set_ylabel('Duration (seconds)')
ax6.set_yscale('log')
ax6.legend(markerscale=3)

plt.suptitle('Order Performance Profiling — Summary Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.savefig(REPORTS_DIR / 'dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dashboard.png")


Saved: dashboard.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65864/1252822420.py:78: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 建議行動方案

### 系統層面 (Layer 1a)
- 將標記的系統異常訂單交 SRE 團隊查修
- 重點排查 device timeout 和 DB lock 問題
- 建立自動告警機制

### User 行為 (Layer 1b)
- 對 contention 頻繁的 device 設置下單頻率限制
- 對大 file_count 訂單設置 warning 或需 approval

### 瓶頸優化 (Layer 2)
- 根據各 file_count 組別的主要瓶頸制定優化策略
- 考慮增加 parallelism (目前 4 threads)

### 慢機台 (Layer 3)
- 優先處理 Top 10 慢機台
- 按 location/system 分組排查共因